# EEGNet Gain and Easy/Hard Subject Analysis

This notebook implements the final additional analysis requested after thesis feedback.

It answers two questions:

1. Can we illustrate easy and hard subjects using raw ERP target/standard plots?
2. Does EEGNet provide more gain for easy subjects or hard subjects?

The main EEGNet gain metric is:

\[
\mathrm{EEGNet\ gain} =
\mathrm{AUC}_{\mathrm{EEGNet}}
-
\max(\mathrm{AUC}_{\mathrm{LDA}}, \mathrm{AUC}_{\mathrm{LogReg}})
\]

A positive value means EEGNet outperformed the best linear model for that subject.
A negative value means the best linear model outperformed EEGNet.

This notebook reads saved result tables and exports final figures/tables under:

`results/thesis_artifacts/eegnet_gain_easy_hard_subjects/`

It does not rerun classifier training.

In [ ]:
# Run from the repository root so that relative result paths resolve
# identically regardless of where this notebook lives.
from oddball_eeg.paths import use_project_root

PROJECT_ROOT = use_project_root()

In [ ]:
from pathlib import Path
import shutil
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import spearmanr, mannwhitneyu

In [ ]:
# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------


RESULTS_DIR = PROJECT_ROOT / "results"

OUT_ROOT = RESULTS_DIR / "thesis_artifacts" / "eegnet_gain_easy_hard_subjects"
FIG_DIR = OUT_ROOT / "figures"
TABLE_DIR = OUT_ROOT / "tables"

FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Results dir :", RESULTS_DIR)
print("Output dir  :", OUT_ROOT)
print("Figure dir  :", FIG_DIR)
print("Table dir   :", TABLE_DIR)

In [ ]:
# ------------------------------------------------------------
# Thesis-ready plot style
# ------------------------------------------------------------

plt.style.use("default")

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,

    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,

    "pdf.fonttype": 42,
    "ps.fonttype": 42,

    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",

    "text.color": "black",
    "axes.labelcolor": "black",
    "axes.edgecolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",

    "axes.grid": False,
})

In [ ]:
# ------------------------------------------------------------
# General helpers
# ------------------------------------------------------------

def find_all(patterns, root=PROJECT_ROOT):
    if isinstance(patterns, str):
        patterns = [patterns]

    matches = []
    for pattern in patterns:
        matches.extend(sorted(root.glob(pattern)))

    return matches


def find_first(patterns, root=PROJECT_ROOT):
    matches = find_all(patterns, root=root)
    return matches[0] if matches else None


def require_file(patterns, description):
    path = find_first(patterns)

    if path is None:
        raise FileNotFoundError(
            f"Could not find required file for: {description}\n"
            "Patterns tried:\n"
            + "\n".join(f"  - {p}" for p in patterns)
        )

    print(f"{description}: {path}")
    return path


def require_columns(df, cols, name):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(
            f"{name} is missing required columns: {missing}\n"
            f"Available columns: {list(df.columns)}"
        )


def clean_dataset_label(x):
    x = str(x)

    replacements = {
        "ds003061": "ds003061",
        "ds005863": "ds005863",
        "ds006466": "ds006466",
        "ds006480": "ds006480 pooled",
        "ds006480_pooled": "ds006480 pooled",
        "ds006480 pooled": "ds006480 pooled",
    }

    return replacements.get(x, x)


def clean_subject_label(x):
    """
    Normalize subject labels while preserving readable subject IDs.
    """
    x = str(x)

    if x.startswith("sub-"):
        return x

    if x.isdigit():
        return f"sub-{int(x):03d}"

    return x


def clean_model_label(x):
    x = str(x)

    replacements = {
        "lda": "LDA",
        "LDA": "LDA",
        "logreg": "Logistic Regression",
        "LogReg": "Logistic Regression",
        "logistic_regression": "Logistic Regression",
        "LogisticRegression": "Logistic Regression",
        "Logistic Regression": "Logistic Regression",
        "eegnet": "EEGNet",
        "EEGNet": "EEGNet",
    }

    return replacements.get(x, x)


def normalize_eval_setting(x):
    x = str(x).strip().lower()

    replacements = {
        "within_subject": "Within-subject",
        "within-subject": "Within-subject",
        "within subject": "Within-subject",
        "grouped_within_subject": "Within-subject",
        "grouped within-subject": "Within-subject",
        "loso": "LOSO",
        "leave_one_subject_out": "LOSO",
        "leave-one-subject-out": "LOSO",
        "leave one subject out": "LOSO",
    }

    return replacements.get(x, x)


def savefig(path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    plt.savefig(path, bbox_inches="tight")
    plt.savefig(path.with_suffix(".png"), bbox_inches="tight")

    print("Saved:", path)
    print("Saved:", path.with_suffix(".png"))

In [ ]:
# ------------------------------------------------------------
# Column detection helpers
# ------------------------------------------------------------

def detect_col(df, candidates, required=True, table_name="table"):
    lower = {c.lower(): c for c in df.columns}

    for c in candidates:
        if c.lower() in lower:
            return lower[c.lower()]

    if required:
        raise ValueError(
            f"Could not detect one of {candidates} in {table_name}.\n"
            f"Available columns: {list(df.columns)}"
        )

    return None


def standardize_auc_table(df, table_name="AUC table"):
    """
    Standardize a subject-level AUC table to columns:

    evaluation_setting, dataset_label, subject, model, auc

    This function accepts several possible original column names.
    """
    df = df.copy()

    eval_col = detect_col(
        df,
        ["evaluation_setting", "setting", "eval_setting", "evaluation"],
        table_name=table_name,
    )

    dataset_col = detect_col(
        df,
        ["dataset_label", "dataset", "dataset_id"],
        table_name=table_name,
    )

    subject_col = detect_col(
        df,
        ["subject", "subject_id", "sub", "participant_id"],
        table_name=table_name,
    )

    model_col = detect_col(
        df,
        ["model", "classifier"],
        table_name=table_name,
    )

    auc_col = detect_col(
        df,
        ["auc", "subject_auc", "mean_auc", "roc_auc"],
        table_name=table_name,
    )

    out = df[[eval_col, dataset_col, subject_col, model_col, auc_col]].copy()
    out.columns = ["evaluation_setting", "dataset_label", "subject", "model", "auc"]

    out["evaluation_setting"] = out["evaluation_setting"].apply(normalize_eval_setting)
    out["dataset_label"] = out["dataset_label"].apply(clean_dataset_label)
    out["subject"] = out["subject"].apply(clean_subject_label)
    out["model"] = out["model"].apply(clean_model_label)

    out = out.dropna(subset=["evaluation_setting", "dataset_label", "subject", "model", "auc"])

    return out

In [ ]:
# ------------------------------------------------------------
# Locate subject-level AUC table
# ------------------------------------------------------------

auc_path = require_file(
    [
        "results/**/auc_with_subject_quality_index.csv",
        "results/**/subject_quality_index_vs_auc_data.csv",
        "results/**/subject_quality*auc*.csv",
        "results/**/relative_importance*/tables/*quality*auc*.csv",
        "results/**/loso_subject_auc*.csv",
        "results/**/subject_auc*.csv",
    ],
    "subject-level AUC / subject-quality table",
)

auc_raw = pd.read_csv(auc_path)
print("Loaded:", auc_path)
display(auc_raw.head())
print("Columns:", list(auc_raw.columns))

In [ ]:
# ------------------------------------------------------------
# Standardize AUC table and preserve quality-related columns
# ------------------------------------------------------------

auc_df = standardize_auc_table(auc_raw, table_name="subject-level AUC table")

# Preserve useful subject-quality/noise/separability columns if they exist.
quality_candidates = [
    "subject_quality_index",
    "quality_index",
    "subject_quality",
    "signal_quality_index",
    "erp_separability",
    "p300_separability",
    "roi_p300_mean_diff",
    "roi_p300_mean_diff_uv",
    "baseline_rms",
    "baseline_rms_uv",
    "baseline_std",
    "baseline_std_uv",
    "poststim_rms",
    "poststim_rms_uv",
]

for col in quality_candidates:
    if col in auc_raw.columns and col not in auc_df.columns:
        auc_df[col] = auc_raw[col].values

# Normalize likely quality column names
rename_map = {}
if "quality_index" in auc_df.columns:
    rename_map["quality_index"] = "subject_quality_index"
if "subject_quality" in auc_df.columns:
    rename_map["subject_quality"] = "subject_quality_index"
if "signal_quality_index" in auc_df.columns:
    rename_map["signal_quality_index"] = "subject_quality_index"

auc_df = auc_df.rename(columns=rename_map)

auc_df.to_csv(TABLE_DIR / "standardized_subject_auc_with_quality.csv", index=False)

display(auc_df.head())
print("Saved:", TABLE_DIR / "standardized_subject_auc_with_quality.csv")
print("Models:", sorted(auc_df["model"].unique()))
print("Evaluation settings:", sorted(auc_df["evaluation_setting"].unique()))
print("Datasets:", sorted(auc_df["dataset_label"].unique()))

In [ ]:
# ------------------------------------------------------------
# Compute EEGNet gain relative to best linear model
# ------------------------------------------------------------

required_models = {"LDA", "Logistic Regression", "EEGNet"}
available_models = set(auc_df["model"].unique())

missing_models = required_models - available_models
if missing_models:
    raise ValueError(
        f"Missing required models for EEGNet gain analysis: {missing_models}\n"
        f"Available models: {available_models}"
    )

pivot = (
    auc_df
    .pivot_table(
        index=["evaluation_setting", "dataset_label", "subject"],
        columns="model",
        values="auc",
        aggfunc="mean",
    )
    .reset_index()
)

pivot["best_linear_auc"] = pivot[["LDA", "Logistic Regression"]].max(axis=1)
pivot["best_linear_model"] = np.where(
    pivot["LDA"] >= pivot["Logistic Regression"],
    "LDA",
    "Logistic Regression",
)

pivot["eegnet_gain_vs_best_linear"] = pivot["EEGNet"] - pivot["best_linear_auc"]
pivot["eegnet_gain_vs_lda"] = pivot["EEGNet"] - pivot["LDA"]
pivot["eegnet_gain_vs_logreg"] = pivot["EEGNet"] - pivot["Logistic Regression"]

# Add subject-quality columns back, one row per subject/evaluation/dataset.
quality_cols = [
    c for c in auc_df.columns
    if c not in ["model", "auc"]
       and c in auc_df.columns
       and c not in ["evaluation_setting", "dataset_label", "subject"]
]

quality_info = (
    auc_df[["evaluation_setting", "dataset_label", "subject"] + quality_cols]
    .drop_duplicates(subset=["evaluation_setting", "dataset_label", "subject"])
)

gain_df = pivot.merge(
    quality_info,
    on=["evaluation_setting", "dataset_label", "subject"],
    how="left",
)

gain_df.to_csv(TABLE_DIR / "eegnet_gain_subject_level.csv", index=False)

display(gain_df.head())
print("Saved:", TABLE_DIR / "eegnet_gain_subject_level.csv")

In [ ]:
# ------------------------------------------------------------
# Identify available quality metrics
# ------------------------------------------------------------

possible_quality_metrics = [
    "subject_quality_index",
    "erp_separability",
    "p300_separability",
    "roi_p300_mean_diff",
    "roi_p300_mean_diff_uv",
]

possible_noise_metrics = [
    "baseline_rms",
    "baseline_rms_uv",
    "baseline_std",
    "baseline_std_uv",
    "poststim_rms",
    "poststim_rms_uv",
]

available_quality_metrics = [c for c in possible_quality_metrics if c in gain_df.columns]
available_noise_metrics = [c for c in possible_noise_metrics if c in gain_df.columns]

print("Available quality metrics:", available_quality_metrics)
print("Available noise metrics  :", available_noise_metrics)

if available_quality_metrics:
    MAIN_EASY_HARD_METRIC = available_quality_metrics[0]
    MAIN_METRIC_DIRECTION = "higher_is_easier"
elif available_noise_metrics:
    MAIN_EASY_HARD_METRIC = available_noise_metrics[0]
    MAIN_METRIC_DIRECTION = "lower_is_easier"
else:
    raise ValueError(
        "No subject-quality, ERP separability, or noise metric found in the AUC table.\n"
        "The EEGNet gain analysis needs at least one easy/hard criterion."
    )

print("Main easy/hard metric:", MAIN_EASY_HARD_METRIC)
print("Direction:", MAIN_METRIC_DIRECTION)

In [ ]:
# ------------------------------------------------------------
# Create easy/hard labels within each dataset and evaluation setting
# ------------------------------------------------------------

def assign_easy_hard(group, metric, direction):
    group = group.copy()

    values = group[metric].astype(float)

    if values.notna().sum() < 6:
        group["easy_hard_group"] = "middle"
        return group

    if direction == "higher_is_easier":
        easy_cut = values.quantile(0.75)
        hard_cut = values.quantile(0.25)

        group["easy_hard_group"] = "middle"
        group.loc[values >= easy_cut, "easy_hard_group"] = "easy"
        group.loc[values <= hard_cut, "easy_hard_group"] = "hard"

    elif direction == "lower_is_easier":
        easy_cut = values.quantile(0.25)
        hard_cut = values.quantile(0.75)

        group["easy_hard_group"] = "middle"
        group.loc[values <= easy_cut, "easy_hard_group"] = "easy"
        group.loc[values >= hard_cut, "easy_hard_group"] = "hard"

    else:
        raise ValueError(f"Unknown direction: {direction}")

    return group


gain_labeled_df = (
    gain_df
    .groupby(["evaluation_setting", "dataset_label"], group_keys=False)
    .apply(assign_easy_hard, metric=MAIN_EASY_HARD_METRIC, direction=MAIN_METRIC_DIRECTION)
    .reset_index(drop=True)
)

gain_labeled_df.to_csv(TABLE_DIR / "eegnet_gain_easy_hard_labeled_subjects.csv", index=False)

display(gain_labeled_df.head())
print("Saved:", TABLE_DIR / "eegnet_gain_easy_hard_labeled_subjects.csv")
print(gain_labeled_df["easy_hard_group"].value_counts())

In [ ]:
# ------------------------------------------------------------
# Summarize EEGNet gain by easy/hard group
# ------------------------------------------------------------

summary_gain = (
    gain_labeled_df
    .query("easy_hard_group in ['easy', 'hard']")
    .groupby(["evaluation_setting", "easy_hard_group"], as_index=False)
    .agg(
        n_subjects=("subject", "count"),
        mean_gain=("eegnet_gain_vs_best_linear", "mean"),
        median_gain=("eegnet_gain_vs_best_linear", "median"),
        std_gain=("eegnet_gain_vs_best_linear", "std"),
        mean_quality=(MAIN_EASY_HARD_METRIC, "mean"),
    )
)

summary_gain.to_csv(TABLE_DIR / "eegnet_gain_easy_hard_summary.csv", index=False)

display(summary_gain)
print("Saved:", TABLE_DIR / "eegnet_gain_easy_hard_summary.csv")

In [ ]:
# ------------------------------------------------------------
# Easy vs hard statistical comparison
# ------------------------------------------------------------

stat_rows = []

for setting, sub in gain_labeled_df.groupby("evaluation_setting"):
    easy = sub.loc[
        sub["easy_hard_group"] == "easy",
        "eegnet_gain_vs_best_linear"
    ].dropna()

    hard = sub.loc[
        sub["easy_hard_group"] == "hard",
        "eegnet_gain_vs_best_linear"
    ].dropna()

    if len(easy) >= 3 and len(hard) >= 3:
        u_stat, p_val = mannwhitneyu(easy, hard, alternative="two-sided")

        stat_rows.append({
            "evaluation_setting": setting,
            "n_easy": len(easy),
            "n_hard": len(hard),
            "mean_gain_easy": easy.mean(),
            "mean_gain_hard": hard.mean(),
            "median_gain_easy": easy.median(),
            "median_gain_hard": hard.median(),
            "difference_mean_easy_minus_hard": easy.mean() - hard.mean(),
            "mannwhitney_u": u_stat,
            "p_value": p_val,
        })

stats_easy_hard = pd.DataFrame(stat_rows)
stats_easy_hard.to_csv(TABLE_DIR / "eegnet_gain_easy_vs_hard_stats.csv", index=False)

display(stats_easy_hard)
print("Saved:", TABLE_DIR / "eegnet_gain_easy_vs_hard_stats.csv")

In [ ]:
# ------------------------------------------------------------
# Correlation between EEGNet gain and subject-quality/noise metrics
# ------------------------------------------------------------

correlation_metrics = available_quality_metrics + available_noise_metrics

corr_rows = []

for metric in correlation_metrics:
    for setting, sub in gain_df.groupby("evaluation_setting"):
        valid = sub[[metric, "eegnet_gain_vs_best_linear"]].dropna()

        if len(valid) < 5:
            continue

        rho, p_val = spearmanr(
            valid[metric],
            valid["eegnet_gain_vs_best_linear"],
        )

        corr_rows.append({
            "evaluation_setting": setting,
            "metric": metric,
            "n": len(valid),
            "spearman_rho": rho,
            "p_value": p_val,
        })

gain_corr_df = pd.DataFrame(corr_rows)
gain_corr_df.to_csv(TABLE_DIR / "eegnet_gain_quality_correlations.csv", index=False)

display(gain_corr_df)
print("Saved:", TABLE_DIR / "eegnet_gain_quality_correlations.csv")

In [ ]:
# ------------------------------------------------------------
# Figure: EEGNet gain vs subject-quality/easy-hard metric
# ------------------------------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

settings = ["Within-subject", "LOSO"]

for ax, setting in zip(axes, settings):
    sub = gain_df[gain_df["evaluation_setting"] == setting].copy()
    sub = sub.dropna(subset=[MAIN_EASY_HARD_METRIC, "eegnet_gain_vs_best_linear"])

    ax.scatter(
        sub[MAIN_EASY_HARD_METRIC],
        sub["eegnet_gain_vs_best_linear"],
        alpha=0.65,
        s=30,
        edgecolor="none",
    )

    if len(sub) >= 5 and sub[MAIN_EASY_HARD_METRIC].std() > 0:
        x = sub[MAIN_EASY_HARD_METRIC].to_numpy()
        y = sub["eegnet_gain_vs_best_linear"].to_numpy()

        slope, intercept = np.polyfit(x, y, 1)
        x_line = np.linspace(x.min(), x.max(), 100)
        y_line = slope * x_line + intercept

        ax.plot(x_line, y_line, linewidth=2)

        rho, p_val = spearmanr(x, y)
        ax.text(
            0.04,
            0.96,
            f"Spearman $\\rho$={rho:.2f}\n$p$={p_val:.3g}",
            transform=ax.transAxes,
            va="top",
            ha="left",
            fontsize=10,
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.8),
        )

    ax.axhline(0, linestyle="--", linewidth=1.2, alpha=0.7)
    ax.set_title(setting)
    ax.set_xlabel(MAIN_EASY_HARD_METRIC.replace("_", " "))
    ax.grid(alpha=0.25)

axes[0].set_ylabel("EEGNet gain vs best linear AUC")

fig.tight_layout()

savefig(FIG_DIR / "fig_eegnet_gain_vs_subject_quality.pdf")
plt.show()

In [ ]:
# ------------------------------------------------------------
# Figure: EEGNet gain in easy vs hard subjects
# ------------------------------------------------------------

plot_df = gain_labeled_df.query("easy_hard_group in ['easy', 'hard']").copy()

plot_df["easy_hard_group"] = pd.Categorical(
    plot_df["easy_hard_group"],
    categories=["hard", "easy"],
    ordered=True,
)

fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharey=True)

for ax, setting in zip(axes, ["Within-subject", "LOSO"]):
    sub = plot_df[plot_df["evaluation_setting"] == setting].copy()

    groups = ["hard", "easy"]
    values = [
        sub.loc[sub["easy_hard_group"] == g, "eegnet_gain_vs_best_linear"].dropna().values
        for g in groups
    ]

    positions = np.arange(len(groups))

    ax.boxplot(
        values,
        positions=positions,
        widths=0.5,
        showfliers=False,
    )

    for pos, vals in zip(positions, values):
        jitter = np.random.default_rng(42).normal(0, 0.045, size=len(vals))
        ax.scatter(
            np.full(len(vals), pos) + jitter,
            vals,
            alpha=0.65,
            s=25,
            edgecolor="none",
        )

    ax.axhline(0, linestyle="--", linewidth=1.2, alpha=0.7)
    ax.set_xticks(positions)
    ax.set_xticklabels(["Hard", "Easy"])
    ax.set_title(setting)
    ax.set_xlabel("Subject group")
    ax.grid(axis="y", alpha=0.25)

axes[0].set_ylabel("EEGNet gain vs best linear AUC")

fig.tight_layout()

savefig(FIG_DIR / "fig_eegnet_gain_easy_vs_hard.pdf")
plt.show()

In [ ]:
# ------------------------------------------------------------
# Choose dataset/evaluation setting for easy-hard ERP illustration
# ------------------------------------------------------------

# Prefer ds003061 if available because it is your original main auditory oddball dataset.
preferred_datasets = ["ds003061", "ds005863", "ds006466", "ds006480 pooled"]

available_datasets = gain_labeled_df["dataset_label"].dropna().unique().tolist()

chosen_dataset = None
for d in preferred_datasets:
    if d in available_datasets:
        chosen_dataset = d
        break

if chosen_dataset is None:
    chosen_dataset = available_datasets[0]

# Prefer within-subject for raw ERP illustration.
chosen_setting = "Within-subject"
if chosen_setting not in gain_labeled_df["evaluation_setting"].unique():
    chosen_setting = gain_labeled_df["evaluation_setting"].iloc[0]

print("Chosen dataset for ERP illustration:", chosen_dataset)
print("Chosen setting for easy/hard selection:", chosen_setting)

In [ ]:
# ------------------------------------------------------------
# Select easy and hard subjects for ERP illustration
# ------------------------------------------------------------

candidate_df = gain_labeled_df[
    (gain_labeled_df["dataset_label"] == chosen_dataset)
    & (gain_labeled_df["evaluation_setting"] == chosen_setting)
    ].copy()

candidate_df = candidate_df.dropna(subset=[MAIN_EASY_HARD_METRIC])

if candidate_df.empty:
    raise ValueError("No candidate subjects found for easy/hard ERP selection.")

if MAIN_METRIC_DIRECTION == "higher_is_easier":
    easy_row = candidate_df.sort_values(MAIN_EASY_HARD_METRIC, ascending=False).iloc[0]
    hard_row = candidate_df.sort_values(MAIN_EASY_HARD_METRIC, ascending=True).iloc[0]
else:
    easy_row = candidate_df.sort_values(MAIN_EASY_HARD_METRIC, ascending=True).iloc[0]
    hard_row = candidate_df.sort_values(MAIN_EASY_HARD_METRIC, ascending=False).iloc[0]

easy_subject = easy_row["subject"]
hard_subject = hard_row["subject"]

selected_subjects_df = pd.DataFrame([
    {
        "role": "easy",
        "dataset_label": chosen_dataset,
        "evaluation_setting": chosen_setting,
        "subject": easy_subject,
        MAIN_EASY_HARD_METRIC: easy_row[MAIN_EASY_HARD_METRIC],
        "eegnet_gain_vs_best_linear": easy_row["eegnet_gain_vs_best_linear"],
        "EEGNet": easy_row["EEGNet"],
        "best_linear_auc": easy_row["best_linear_auc"],
    },
    {
        "role": "hard",
        "dataset_label": chosen_dataset,
        "evaluation_setting": chosen_setting,
        "subject": hard_subject,
        MAIN_EASY_HARD_METRIC: hard_row[MAIN_EASY_HARD_METRIC],
        "eegnet_gain_vs_best_linear": hard_row["eegnet_gain_vs_best_linear"],
        "EEGNet": hard_row["EEGNet"],
        "best_linear_auc": hard_row["best_linear_auc"],
    },
])

selected_subjects_df.to_csv(TABLE_DIR / "selected_easy_hard_subjects_for_erp.csv", index=False)

display(selected_subjects_df)
print("Saved:", TABLE_DIR / "selected_easy_hard_subjects_for_erp.csv")

In [ ]:
# ------------------------------------------------------------
# Locate subject-level ERP waveform table
# ------------------------------------------------------------

erp_subject_path = find_first([
    "results/**/subject_erp_waveforms_long.csv",
    "results/**/erp_subject_waveforms_long.csv",
    "results/**/erp*subject*waveform*.csv",
    "results/**/subject*waveform*.csv",
])

print("Subject ERP waveform table:", erp_subject_path)

if erp_subject_path is None:
    warnings.warn(
        "No subject-level ERP waveform table found. "
        "The easy/hard ERP figure cannot be generated from data yet. "
        "See the next markdown cell for the export block to add to your ERP sanity notebook."
    )
else:
    erp_subject_df = pd.read_csv(erp_subject_path)
    print("Loaded:", erp_subject_path)
    display(erp_subject_df.head())
    print("Columns:", list(erp_subject_df.columns))

## If the subject-level ERP table is missing

If the previous cell did not find a table, go back to the ERP sanity notebook and export a subject-level ERP table with columns like:

- `dataset_label`
- `subject`
- `time_s`
- `standard_uv`
- `target_uv`
- `difference_uv`

The final easy/hard ERP figure needs subject-level waveforms, not just dataset-level grand averages.

In [ ]:
# ------------------------------------------------------------
# Standardize subject-level ERP waveform table
# ------------------------------------------------------------

if erp_subject_path is not None:
    df = erp_subject_df.copy()
    lower = {c.lower(): c for c in df.columns}

    # Dataset column is optional here.
    # Some subject-level ERP exports are for one dataset only and therefore
    # contain subject/time/waveform columns but no dataset column.
    dataset_col = detect_col(
        df,
        ["dataset_label", "dataset", "dataset_id"],
        required=False,
        table_name="subject ERP waveform table",
    )

    subject_col = detect_col(
        df,
        ["subject", "subject_id", "sub", "participant_id"],
        table_name="subject ERP waveform table",
    )

    time_col = detect_col(
        df,
        ["time_s", "time", "times", "latency_s"],
        table_name="subject ERP waveform table",
    )

    standard_col = detect_col(
        df,
        ["standard_uv", "standard", "standard_mean_uv"],
        required=False,
        table_name="subject ERP waveform table",
    )

    target_col = detect_col(
        df,
        ["target_uv", "target", "target_mean_uv"],
        required=False,
        table_name="subject ERP waveform table",
    )

    diff_col = detect_col(
        df,
        ["difference_uv", "difference", "diff_uv", "diff", "target_minus_standard_uv"],
        required=False,
        table_name="subject ERP waveform table",
    )

    # Build standardized dataframe.
    if dataset_col is not None:
        erp_easyhard_df = df[[dataset_col, subject_col, time_col]].copy()
        erp_easyhard_df.columns = ["dataset_label", "subject", "time_s"]
    else:
        erp_easyhard_df = df[[subject_col, time_col]].copy()
        erp_easyhard_df.columns = ["subject", "time_s"]
        erp_easyhard_df["dataset_label"] = chosen_dataset
        print(
            "No dataset column found in subject ERP waveform table. "
            f"Assigning dataset_label = {chosen_dataset!r}."
        )

    if standard_col is not None:
        erp_easyhard_df["standard_uv"] = df[standard_col].values

    if target_col is not None:
        erp_easyhard_df["target_uv"] = df[target_col].values

    if diff_col is not None:
        erp_easyhard_df["difference_uv"] = df[diff_col].values

    if "difference_uv" not in erp_easyhard_df.columns:
        if {"standard_uv", "target_uv"}.issubset(erp_easyhard_df.columns):
            erp_easyhard_df["difference_uv"] = (
                    erp_easyhard_df["target_uv"] - erp_easyhard_df["standard_uv"]
            )
        else:
            raise ValueError(
                "Subject ERP waveform table needs either difference/difference_uv "
                "or both standard and target columns."
            )

    erp_easyhard_df["dataset_label"] = erp_easyhard_df["dataset_label"].apply(clean_dataset_label)
    erp_easyhard_df["subject"] = erp_easyhard_df["subject"].apply(clean_subject_label)

    # Reorder columns for readability.
    ordered_cols = ["dataset_label", "subject", "time_s"]
    for c in ["standard_uv", "target_uv", "difference_uv"]:
        if c in erp_easyhard_df.columns:
            ordered_cols.append(c)

    erp_easyhard_df = erp_easyhard_df[ordered_cols].copy()

    # Convert values to microvolts if needed.
    value_cols = [
        c for c in ["standard_uv", "target_uv", "difference_uv"]
        if c in erp_easyhard_df.columns
    ]

    max_abs = erp_easyhard_df[value_cols].abs().max().max()

    if max_abs < 1e-3:
        for c in value_cols:
            erp_easyhard_df[c] = erp_easyhard_df[c] * 1e6
        print("Converted subject ERP values from volts to microvolts.")
    else:
        print("Subject ERP values appear to already be in microvolts.")

    erp_easyhard_df.to_csv(
        TABLE_DIR / "standardized_subject_erp_waveforms_long.csv",
        index=False,
    )

    display(erp_easyhard_df.head())
    print("Saved:", TABLE_DIR / "standardized_subject_erp_waveforms_long.csv")

In [ ]:
# ------------------------------------------------------------
# Figure: Easy vs hard subject raw ERP waveforms
# ------------------------------------------------------------

if erp_subject_path is not None:
    plot_erp = erp_easyhard_df[
        (erp_easyhard_df["dataset_label"] == chosen_dataset)
        & (erp_easyhard_df["subject"].isin([easy_subject, hard_subject]))
        ].copy()

    if plot_erp.empty:
        raise ValueError(
            f"No ERP waveform rows found for selected subjects: {easy_subject}, {hard_subject}"
        )

    role_map = {
        easy_subject: "Easy subject",
        hard_subject: "Hard subject",
    }

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharey=False)

    for ax, subject in zip(axes, [easy_subject, hard_subject]):
        sub = plot_erp[plot_erp["subject"] == subject].sort_values("time_s")

        role = role_map[subject]

        if {"standard_uv", "target_uv"}.issubset(sub.columns):
            ax.plot(
                sub["time_s"],
                sub["standard_uv"],
                linewidth=2,
                label="Standard",
            )
            ax.plot(
                sub["time_s"],
                sub["target_uv"],
                linewidth=2,
                label="Target",
            )
        else:
            ax.plot(
                sub["time_s"],
                sub["difference_uv"],
                linewidth=2,
                label="Target - standard",
            )

        ax.axvline(0, linestyle="--", linewidth=1, alpha=0.6)
        ax.axhline(0, linestyle="--", linewidth=1, alpha=0.4)

        metric_value = selected_subjects_df.loc[
            selected_subjects_df["subject"] == subject,
            MAIN_EASY_HARD_METRIC,
        ].iloc[0]

        ax.set_title(
            f"{role}: {subject}\n"
            f"{MAIN_EASY_HARD_METRIC.replace('_', ' ')} = {metric_value:.3f}"
        )

        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Amplitude (µV)")
        ax.grid(alpha=0.25)
        ax.legend(frameon=False)

    fig.suptitle(f"Easy and hard subject ERP examples from {chosen_dataset}", y=1.04)
    fig.tight_layout()

    savefig(FIG_DIR / "fig_easy_hard_subject_raw_erp_examples.pdf")
    plt.show()

In [ ]:
# ------------------------------------------------------------
# Generate concise interpretation text for dissertation
# ------------------------------------------------------------

lines = []

lines.append("EEGNet gain analysis summary")
lines.append("=" * 40)
lines.append("")
lines.append(f"Easy/hard criterion: {MAIN_EASY_HARD_METRIC}")
lines.append(f"Direction: {MAIN_METRIC_DIRECTION}")
lines.append("")

if not summary_gain.empty:
    lines.append("Mean EEGNet gain by easy/hard group:")
    for _, row in summary_gain.iterrows():
        lines.append(
            f"- {row['evaluation_setting']}, {row['easy_hard_group']}: "
            f"mean gain={row['mean_gain']:.4f}, "
            f"median gain={row['median_gain']:.4f}, "
            f"n={int(row['n_subjects'])}"
        )
    lines.append("")

if not gain_corr_df.empty:
    lines.append("Spearman correlations between EEGNet gain and subject metrics:")
    for _, row in gain_corr_df.iterrows():
        lines.append(
            f"- {row['evaluation_setting']}, {row['metric']}: "
            f"rho={row['spearman_rho']:.3f}, "
            f"p={row['p_value']:.4g}, n={int(row['n'])}"
        )
    lines.append("")

if not stats_easy_hard.empty:
    lines.append("Easy-vs-hard Mann-Whitney comparisons:")
    for _, row in stats_easy_hard.iterrows():
        lines.append(
            f"- {row['evaluation_setting']}: "
            f"mean easy gain={row['mean_gain_easy']:.4f}, "
            f"mean hard gain={row['mean_gain_hard']:.4f}, "
            f"easy-hard difference={row['difference_mean_easy_minus_hard']:.4f}, "
            f"p={row['p_value']:.4g}"
        )

summary_text = "\n".join(lines)

summary_path = OUT_ROOT / "eegnet_gain_easy_hard_summary.txt"
summary_path.write_text(summary_text)

print(summary_text)
print("\nSaved:", summary_path)

In [ ]:
# ------------------------------------------------------------
# Save artifact manifest
# ------------------------------------------------------------

manifest_rows = []

for path in sorted(FIG_DIR.glob("*")):
    manifest_rows.append({
        "artifact_type": "figure",
        "filename": path.name,
        "relative_path": str(path.relative_to(PROJECT_ROOT)),
        "size_kb": round(path.stat().st_size / 1024, 1),
    })

for path in sorted(TABLE_DIR.glob("*")):
    manifest_rows.append({
        "artifact_type": "table",
        "filename": path.name,
        "relative_path": str(path.relative_to(PROJECT_ROOT)),
        "size_kb": round(path.stat().st_size / 1024, 1),
    })

summary_file = OUT_ROOT / "eegnet_gain_easy_hard_summary.txt"
if summary_file.exists():
    manifest_rows.append({
        "artifact_type": "summary",
        "filename": summary_file.name,
        "relative_path": str(summary_file.relative_to(PROJECT_ROOT)),
        "size_kb": round(summary_file.stat().st_size / 1024, 1),
    })

manifest_df = pd.DataFrame(manifest_rows)

manifest_path = OUT_ROOT / "eegnet_gain_easy_hard_manifest.csv"
manifest_df.to_csv(manifest_path, index=False)

display(manifest_df)
print("Saved:", manifest_path)